#### Packages

In [37]:
import xarray as xr
from tqdm import tqdm
import numpy as np
import calendar
from datetime import datetime
import pandas as pd
import ast
from cartopy.io import shapereader

In [38]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

In [39]:
import psutil
import dask
from dask.distributed import Client
client = Client(
    n_workers=20
)
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42737 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/42737/status,
Dashboard: /proxy/42737/status,Workers: 20
Total threads: 20,Total memory: 20.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44093,Workers: 0
Dashboard: /proxy/42737/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:45635,Total threads: 1
Dashboard: /proxy/42965/status,Memory: 1.00 GiB
Nanny: tcp://127.0.0.1:45267,


2026-06-18 15:32:00,309 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:44759' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('shuffle-split-68d2194b738d4fa97820c0c2cced7571', 785), ('shuffle-split-68d2194b738d4fa97820c0c2cced7571', 71), ('concatenate-getitem-transpose-e2707a1f91f794f4b98cb815f85ec9a5', 2, 1, 13), 'shuffle-taker-cd31c8fefd81a1798bf6831b9b5179d4', ('getitem-daf82375d1e4ff0a5f19446b9479400d', 3, 2, 1), ('shuffle-split-68d2194b738d4fa97820c0c2cced7571', 443), ('shuffle-split-68d2194b738d4fa97820c0c2cced7571', 1121), ('shuffle-split-d5af15340fbd815b05e0ce55a44a804a', 31), ('array-4162ed96693e72083c44677ae389f8e7', 24)} (stimulus_id='handle-worker-cleanup-1781760720.3089604')
2026-06-18 15:32:00,627 - distributed.nanny - WARNING - Restarting worker
2026-06-18 15:32:03,526 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:45635' caused the cluster to lose already computed task(s), which will 

#### Functions

#### Main

In [43]:
# directories
workdir = '/g/data/if69/cj0591/GC26_energy_synoptics'
datadir = f'{workdir}/data/weatherfeatures.era5'

# era5 is under rt52
era5dir = '/g/data/rt52/era5/single-levels'

In [44]:
df = pd.read_csv('/scratch/if69/cj0591/State_extreme_demand_dates_2010_2016.csv')

In [70]:
state = 'VIC'

In [71]:
highdemand = df[df['state']==state]['high_dates'].to_numpy()
highdemand = [
    d.strip()
    for d in highdemand[0].split(",")
    if d.strip()
]

lowdemand = df[df['state']==state]['low_dates'].to_numpy()
lowdemand = [
    d.strip()
    for d in lowdemand[0].split(",")
    if d.strip()
]

In [47]:
def create_datetime_string(year, month):
    if 1 <= month <= 12:
        last_day = calendar.monthrange(year, month)[1]
        first_date = datetime(year, month, 1)
        last_date = datetime(year, month, last_day)
        return f"{first_date.strftime('%Y%m%d')}-{last_date.strftime('%Y%m%d')}"
    else:
        return "Invalid month"

In [50]:
years = range(2010, 2017) 
months = [6, 7, 8]    

u10files = []
v10files = []
for year in years:
    for month in months:
        date_string = create_datetime_string(year, month)

        u10file = (
            f"{ERA5_SFC_DIR}/10u/{date_string[0:4]}/"
            f"10u_era5_oper_sfc_{date_string}.nc"
        )

        v10file = (
            f"{ERA5_SFC_DIR}/10v/{date_string[0:4]}/"
            f"10v_era5_oper_sfc_{date_string}.nc"
        )

        u10files.append(u10file)
        v10files.append(v10file)

In [52]:
def preprocess(ds):
    ds = ds.sel(longitude=slice(100, 155), latitude=slice(-9, -55))
    ds = ds.sel(time=ds.time.dt.hour.isin([0, 6, 12, 18]))
    return ds

In [53]:
dsu = xr.open_mfdataset(u10files, parallel=True, preprocess=preprocess)
dsv = xr.open_mfdataset(v10files, parallel=True, preprocess=preprocess)

In [54]:
# daily mean
dsu_daily = dsu.resample(time="1D").mean()
dsv_daily = dsv.resample(time="1D").mean()

In [ ]:
# climatology
dsu_daily_clima = dsu_daily.mean('time').compute()
dsv_daily_clima = dsv_daily.mean('time').compute()

In [61]:
# anomaly
dsu_anomaly = dsu_daily - dsu_daily_clima
dsv_anomaly = dsv_daily - dsv_daily_clima